# CURB - Emerging-Hotspot Trends
### notebook 04 - Problem Statement 1 (Parking-Induced Congestion)

The defensible **reactive -> proactive** layer: which spots got *worse* over the data
window, so enforcement can get ahead of them.

**Why this is trustworthy here:** citywide weekly volume is flat across the period (we
check this below - the second/first-half ratio is ~1.0). So a cell that rises is genuinely
rising, not riding a citywide ramp. We trim the two partial boundary weeks, split the rest
into first vs second half, and measure each cell's growth **relative to the city**.

**Honesty (for the deck):** this is a *retrospective* trend over ~5 months, not a future
forecast. Dates were shifted by a constant offset in anonymisation (relative trends valid,
absolute dates not); hour-of-day is unreliable and unused. Provided dataset only.

In [ ]:
import os, sys, json
from pathlib import Path
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config.py").exists() and (REPO_ROOT.parent / "config.py").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT)); os.chdir(REPO_ROOT)
import config, pandas as pd, matplotlib.pyplot as plt
from src.trends import run
DATA_PATH = config.DATA_PATH
pd.set_option("display.max_colwidth", 34)
print("data:", DATA_PATH, "exists:", os.path.exists(DATA_PATH))

## 1 - Run + validity check

In [ ]:
cell, emerging, city_ratio = run(DATA_PATH)
print(f"Citywide second/first-half ratio: {city_ratio:.3f}  (near 1.0 => flat baseline, trends are real)")
print(f"{len(cell):,} cells assessed (>=20 citations)")
print(cell.trend.value_counts().to_string())

## 2 - The trend split

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
cell.trend.value_counts().reindex(["Emerging","Stable","Cooling"]).plot.bar(
    ax=ax[0], color=["#E5564B","#7C8DA6","#4C9BE6"])
ax[0].set_title("Cells by trend"); ax[0].set_ylabel("cells")
ax[1].scatter(cell["first"], cell["second"], s=12,
              c=cell.trend.map({"Emerging":"#E5564B","Stable":"#7C8DA6","Cooling":"#4C9BE6"}), alpha=.5)
lim = max(cell["first"].max(), cell["second"].max())
ax[1].plot([0, lim], [0, lim], "--", color="#5E6F8A", linewidth=1)
ax[1].set_xlabel("first half (citations)"); ax[1].set_ylabel("second half")
ax[1].set_title("First vs second half (above line = rising)")
plt.tight_layout(); plt.show()

## 3 - Top emerging hotspots (biggest real rise)

In [ ]:
emerging.head(12)[["rank","name","first","second","change","rel_growth","total","impact_score"]]

### The priority watchlist: rising **and** already high-impact
Sort the emerging spots by impact - these aren't just trending up, they already carry real
obstruction burden. This is the "get ahead of it" list.

In [ ]:
emerging.sort_values("impact_score", ascending=False).head(10)[
    ["name","first","second","change","rel_growth","impact_score"]]

## 4 - Cooling spots (for contrast)\nSome big spots are improving - useful to show enforcement effort isn't static.

In [ ]:
cell[cell.trend == "Cooling"].sort_values("change").head(10)[
    ["name","first","second","change","rel_growth","impact_score"]]

## 5 - Files written

In [ ]:
for f in ["outputs/curb_trends.json","outputs/curb_trends.csv"]:
    print("wrote", f, "-", os.path.getsize(f), "bytes")
j = json.load(open("outputs/curb_trends.json"))
print("\nmeta:", json.dumps(j["meta"], indent=2))

## Methodology note
- Trend = second-half vs first-half citations per cell, over the trimmed window
  (partial start/end weeks removed), normalised against the flat citywide baseline.
- "Emerging" requires rising >1.5x the citywide rate AND a minimum absolute increase, so
  tiny noisy cells are excluded (thresholds are documented constants in `src/trends.py`).
- Retrospective trend, not a forecast. Provided dataset only; hour-of-day not used.